Ce notebook a pour objectif de mettre en pratique notre modele Random Forest deja entraine en l'appliquant a de nouveaux clients, afin de predire leurs probabilites de souscription, de les classer selon leur potentiel et d'identifier ceux a cibler lors d'une campagne marketing.
Maintenant que notre modele a ete entraine, il est temps de le mettre dans un milieu plus realiste pour voir ses performances.

In [19]:
import pandas as pd
import joblib
import numpy as np

model = joblib.load("../models/random_forest.pkl")
new_clients = pd.read_csv("../data/raw/new_clients.csv")

In [20]:
print(model) 
print("Dimensions :", new_clients.shape)
print("\nColonnes :")
print(new_clients.columns.tolist())
print("\nAperçu :")
new_clients.head()
# juste pour s'assurer que l'import c'est bien passé

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'balance', 'day',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome'])])),
                ('model',
                 RandomForestClassifier(n_estimators=300, n_jobs=-1,
                                        random_state=42))])
Dimensions : (500, 15)

Colonnes :
['age', 'job', 'marital', 'education', '

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome
0,27,management,single,tertiary,no,-31,no,no,cellular,3,mar,1,-1,0,unknown
1,53,retired,married,tertiary,no,131,no,no,cellular,18,aug,5,-1,0,unknown
2,36,technician,married,secondary,no,631,no,no,cellular,8,aug,13,-1,0,unknown
3,28,self-employed,single,tertiary,no,904,no,no,cellular,19,nov,2,33,4,success
4,47,management,married,tertiary,no,310,yes,no,cellular,22,jul,2,-1,0,unknown


In [21]:
probabilities = model.predict_proba(new_clients)[:, 1]
print(probabilities[:10])
print("Nombre de probabilités :", len(probabilities))
print("Probabilité minimale :", probabilities.min())
print("Probabilité maximale :", probabilities.max())

[0.77666667 0.40666667 0.26333333 0.86333333 0.26333333 0.05333333
 0.63333333 0.87666667 0.2        0.38      ]
Nombre de probabilités : 500
Probabilité minimale : 0.023333333333333334
Probabilité maximale : 0.9933333333333333


OK, checkpoint, tout fonctionne bien, on obtient bien une probabilite par client, qui va donc de 20% a 99%. Au vu de notre echantillon de probabilites, on peut voir qu'elles semblent assez variees, ce qui est interesse pour la suite.
Du coup, on va maintenant ajouter la colonne de proba de souscription sur notre dataframe :

In [22]:
new_clients["subscription_probability"] = probabilities
new_clients.head(10) # pour s'assurer que tout soit ok

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,subscription_probability
0,27,management,single,tertiary,no,-31,no,no,cellular,3,mar,1,-1,0,unknown,0.776667
1,53,retired,married,tertiary,no,131,no,no,cellular,18,aug,5,-1,0,unknown,0.406667
2,36,technician,married,secondary,no,631,no,no,cellular,8,aug,13,-1,0,unknown,0.263333
3,28,self-employed,single,tertiary,no,904,no,no,cellular,19,nov,2,33,4,success,0.863333
4,47,management,married,tertiary,no,310,yes,no,cellular,22,jul,2,-1,0,unknown,0.263333
5,55,services,married,secondary,no,-276,yes,no,unknown,27,may,12,-1,0,unknown,0.053333
6,24,management,single,tertiary,no,794,yes,no,cellular,16,jul,1,-1,0,unknown,0.633333
7,33,admin.,single,secondary,no,1148,yes,no,cellular,1,jun,1,183,2,success,0.876667
8,27,admin.,married,secondary,no,6019,yes,yes,cellular,9,may,1,352,2,failure,0.200000
9,39,blue-collar,married,primary,no,655,no,no,cellular,27,apr,3,-1,0,unknown,0.380000


tout semble ok, on a bien notre colonne qui vient s'ajouter a la suite des autres !
on va donc maintenant trier notre csv pour que l'on ait en tete les clients les plus susceptibles de souscrire, jusq'au moins susceptible de souscrire.
D'un point de vue metier, on pourra par la suite transferer ce csv trie a une autre equipe de communication par exemple, qui jugera en fonction du budget accorde a cette campagne le nombre de client a contacte, et il sera donc plus facile pour eux de retrouver ces clients si notre tableau est bien trie.

In [23]:
new_clients = new_clients.sort_values(
    by="subscription_probability",
    ascending=False
)

In [24]:
new_clients.head(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,subscription_probability
66,42,management,married,tertiary,no,949,no,no,cellular,8,sep,1,183,5,success,0.993333
316,33,management,single,tertiary,no,3023,no,no,cellular,17,mar,3,187,3,success,0.976667
144,30,student,single,tertiary,no,856,no,no,cellular,14,oct,1,183,4,success,0.970000
154,66,retired,married,secondary,no,2374,no,yes,cellular,15,feb,1,181,1,success,0.970000
416,26,services,single,secondary,no,932,no,no,cellular,21,dec,1,176,4,success,0.963333
169,55,management,divorced,tertiary,no,5182,no,no,cellular,15,apr,1,87,1,success,0.960000
224,78,retired,married,primary,no,3163,no,no,cellular,16,nov,1,92,3,success,0.956667
295,35,unemployed,single,tertiary,no,226,no,no,cellular,10,may,2,89,2,success,0.953333
41,37,management,divorced,tertiary,no,3210,no,no,cellular,13,aug,3,43,8,other,0.953333
455,53,admin.,married,secondary,no,984,no,no,telephone,31,mar,1,91,2,success,0.953333


Dans ce nouveau dataset, certains clients possedent la valeur "success" pour la variable "poutcome". Cela signifie qu'ils ont deja souscrit a un produit lors d'une precedente campagne marketing. Il ne s'agit pas necessairement de nouveaux clients de la banque, mais de nouvelles observations qui n'ont pas ete utilisee lors de l'entrainement du modele.
une banque peut tout a fait recontacter des clients existants pour leur proposer de nouveaux produits ou services !

In [ ]:
threshold = 0.4

new_clients["prediction"] = np.where(
    new_clients["subscription_probability"] >= threshold,
    "yes",
    "no"
)

new_clients["prediction"].value_counts()
new_clients["prediction"].value_counts(normalize=True) * 100 # permet juste de voir le pourcentage de chaque classe

prediction
yes    57.8
no     42.2
Name: proportion, dtype: float64

ok maintenant qu'on a tout ca, comme dit precedemment, on va creer notre tableau des clients cibles, ceux a contacter. C'est toujours mieux de faire un autre tableau que l'original, qui sera lui seulement destine a la campgne marketing, tandis que l'original sert pour notre model. Ce dernier ne sera pas re injecter, mais sera juste transmis a d'autres equipes pour un but marketing.

In [28]:
targeted_clients = new_clients[
    new_clients["prediction"] == "yes"
].copy()
targeted_clients.head(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,subscription_probability,prediction
66,42,management,married,tertiary,no,949,no,no,cellular,8,sep,1,183,5,success,0.993333,yes
316,33,management,single,tertiary,no,3023,no,no,cellular,17,mar,3,187,3,success,0.976667,yes
144,30,student,single,tertiary,no,856,no,no,cellular,14,oct,1,183,4,success,0.970000,yes
154,66,retired,married,secondary,no,2374,no,yes,cellular,15,feb,1,181,1,success,0.970000,yes
416,26,services,single,secondary,no,932,no,no,cellular,21,dec,1,176,4,success,0.963333,yes
169,55,management,divorced,tertiary,no,5182,no,no,cellular,15,apr,1,87,1,success,0.960000,yes
224,78,retired,married,primary,no,3163,no,no,cellular,16,nov,1,92,3,success,0.956667,yes
295,35,unemployed,single,tertiary,no,226,no,no,cellular,10,may,2,89,2,success,0.953333,yes
41,37,management,divorced,tertiary,no,3210,no,no,cellular,13,aug,3,43,8,other,0.953333,yes
455,53,admin.,married,secondary,no,984,no,no,telephone,31,mar,1,91,2,success,0.953333,yes


et donc pour conclure, on finit par importer notre tableau en format csv :

In [29]:
targeted_clients.to_csv(
    "../data/processed/targeted_clients.csv",
    index=False
)

ne pas oublier d'aller verifier si l'export s'est bien passé !

Pour conclure cette phase secondaire, il est important de souligner qu'elle reste necessaire a notre projet pour pouvoir l'apporter dans un milieu encore plus professionel et ne pas laisser nos modeles et donnees dans un simple projet personnel. Il est en effet bien plus interesse de se servir de ce qu'on a cree pour le voir se mettre en oeuvre et s'en servir reelement, dans les prochaines phases on pourra donc analyser de nouvelles donnees avec de nouveaux clients, ce qui rend le projet encore plus realiste !

Bilan : notre modele est desormais capable de recevoir de nouvelles donnees, de produire des probabilites de souscription et de generer une liste de clients prioritaires pour une campagne marketing !